# PyTorch 快速入门 —— 强化学习前置

本 Notebook 从 tensor 基础讲到神经网络搭建与训练，均为可运行的入门代码。
后续 Q-learning 延伸（DQN、Policy Gradient）将直接依赖这些内容。

**前提**：已了解 Python 基础（数据类型、函数、类）。
**使用方式**：按顺序运行每个单元格（Shift + Enter）。

## 1. Tensor 基础

Tensor 是 PyTorch 的核心数据结构，类似 NumPy 的 ndarray，但支持 GPU 加速和自动求导。

In [ ]:
import torch
import numpy as np

# ---- 创建 Tensor ----
a = torch.tensor([1, 2, 3])              # 从列表
b = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # 2D
c = torch.zeros(3, 4)                    # 全 0
d = torch.ones(2, 3)                     # 全 1
e = torch.rand(2, 3)                     # [0,1) 均匀分布
f = torch.randn(2, 3)                    # 标准正态
g = torch.arange(0, 10, 2)               # 等差数列

# 从 NumPy 互转
arr = np.array([1, 2, 3])
t_from_np = torch.from_numpy(arr)        # NumPy -> Tensor (共享内存)
t_to_np = t_from_np.numpy()              # Tensor -> NumPy

print("形状:", b.shape, "  维度数:", b.ndim, "  元素数:", b.numel())
print("dtype:", b.dtype)

## 2. Tensor 运算

语法与 NumPy 高度一致，支持广播、索引、切片。

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([4.0, 5.0, 6.0])

# 逐元素
print("x + y =", x + y)
print("x * y =", x * y)
print("x ** 2 =", x ** 2)

# 矩阵运算
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])
print("矩阵乘 A @ B:\n", A @ B)         # 等价于 torch.mm(A, B)
print("逐元素乘 A * B:\n", A * B)       # Hadamard 积

# 广播
vec = torch.tensor([10.0, 20.0])
print("A + vec (广播):\n", A + vec)

In [ ]:
# 索引与切片（与 NumPy 一致）
t = torch.arange(12).reshape(3, 4)
print("原始:\n", t)
print("第0行:", t[0])
print("最后一列:", t[:, -1])
print("前2行 后2列:\n", t[:2, -2:])

# 布尔索引
mask = t > 5
print(">5 的元素:", t[mask])

# 形状变换
print("reshape(6,2):\n", t.reshape(6, 2))
print("转置:\n", t.T)

# 常用聚合
print("sum:", t.sum(), "mean:", t.float().mean(), "max:", t.max())

## 3. 设备：CPU vs GPU

RL 中大量数据并行计算，GPU 可显著加速。

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 创建时指定设备
t = torch.tensor([1.0, 2.0], device=device)

# 或在现有 tensor 上切换
t_cpu = torch.tensor([3.0, 4.0])
t_gpu = t_cpu.to(device)           # 移到 GPU
t_back = t_gpu.cpu()               # 移回 CPU

print("tensor 所在设备:", t.device)

# 注意：不同设备上的 tensor 不能直接运算
# t + t_cpu  # 会报错！

## 4. 自动求导 (autograd)

PyTorch 可以自动计算梯度，这是训练神经网络的核心。

In [ ]:
# requires_grad=True 表示需要计算梯度
x = torch.tensor(2.0, requires_grad=True)

# 前向计算: y = 3x^2 + 2x + 1
y = 3 * x ** 2 + 2 * x + 1

# 反向传播，计算 dy/dx
y.backward()

# x.grad 即 dy/dx = 6x + 2 = 6*2+2 = 14
print(f"x = {x.item()}, y = {y.item()}, dy/dx = {x.grad.item()}")

In [ ]:
# 多变量梯度
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# loss = sum(w_i^2)
loss = (w ** 2).sum()
loss.backward()

# d(loss)/dw_i = 2*w_i
print("w 的梯度:", w.grad)  # [2, 4, 6]

# 注意：每次 backward 前需要清零梯度
w.grad.zero_()

In [ ]:
# 梯度传递链
a = torch.tensor(1.0, requires_grad=True)
b = 2 * a          # b = 2a,  db/da = 2
c = 3 * b          # c = 6a,  dc/da = 6
d = c ** 2         # d = 36a^2, dd/da = 72a = 72
d.backward()
print(f"a.grad = {a.grad.item()}")  # 72

# 检查中间变量的梯度 (默认不保留，需要 retain_grad)
print(f"b.grad = {b.grad}")  # None — 中间节点不保存梯度

In [ ]:
# 不需要梯度时的写法（推理/评估时节省内存）
with torch.no_grad():
    x = torch.tensor(3.0, requires_grad=True)
    y = x ** 2
    print("no_grad 下 y 不追踪:", y.requires_grad)  # False

# 也可以用 .detach() 切断梯度图
z = y.detach()
print("detach 后:", z.requires_grad)  # False

## 5. 构建神经网络：nn.Module

继承 `nn.Module`，在 `__init__` 中定义层，在 `forward` 中定义前向传播。

In [ ]:
import torch.nn as nn

# 一个简单的全连接网络
class SimpleNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)   # 全连接层
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()                         # 激活函数
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)              # 输出层通常不加激活
        return x

# 实例化
net = SimpleNet(input_dim=4, hidden_dim=64, output_dim=2)
print(net)
print(f"参数数量: {sum(p.numel() for p in net.parameters())}")

### 5.1 常用层一览

### 5.2 激活函数

激活函数赋予网络非线性表达能力。没有它们，多层网络退化为单层线性变换。
以下是 PyTorch 中常用激活函数的公式与对比。

In [ ]:
import matplotlib.pyplot as plt

x = torch.linspace(-4, 4, 200)
activations = {
    "Sigmoid":      torch.sigmoid(x),              # 1/(1+e^{-x})  — 输出在 (0,1)，适合二分类输出层
    "Tanh":         torch.tanh(x),                 # 输出在 (-1,1)，零中心，RNN 常用
    "ReLU":         torch.relu(x),                 # max(0,x) — 计算最快，最常用的隐藏层激活
    "LeakyReLU":    torch.nn.functional.leaky_relu(x, 0.1),  # 负半轴有小斜率，缓解 ReLU 死亡
    "SiLU / Swish": torch.nn.functional.silu(x),   # x*sigmoid(x) — 光滑版 ReLU，大模型常用
    "GELU":         torch.nn.functional.gelu(x),   # x*Phi(x) — Transformer / BERT 标准激活
}

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (name, y) in zip(axes.flat, activations.items()):
    ax.plot(x.numpy(), y.numpy(), linewidth=2)
    ax.set_title(name)
    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.5)
plt.suptitle("Activation Functions", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Sequential — 快速堆叠层
tiny_net = nn.Sequential(
    nn.Linear(4, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
)

# 输入 batch=3, 特征=4
x = torch.randn(3, 4)
out = tiny_net(x)
print(f"输入形状: {x.shape} -> 输出形状: {out.shape}")
print("输出:\n", out)

In [ ]:
# 常用层示例
batch, seq_len, features = 2, 5, 8

# Conv2d — 卷积层
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)

# BatchNorm — 批归一化
bn = nn.BatchNorm1d(features)

# Dropout — 随机丢弃（防止过拟合）
dropout = nn.Dropout(p=0.5)  # train 时生效，eval 时自动关闭

# Embedding — 整数索引 -> 稠密向量
embed = nn.Embedding(num_embeddings=10, embedding_dim=8)

indices = torch.tensor([1, 3, 5])
print("Embedding 输出形状:", embed(indices).shape)  # (3, 8)

## 6. 损失函数与优化器

训练 = 前向传播 -> 计算 loss -> 反向传播 -> 更新参数。

In [ ]:
import torch.optim as optim

# 准备一个简单网络和数据
net = SimpleNet(input_dim=4, hidden_dim=32, output_dim=1)

# 损失函数（MSE 适合回归，CrossEntropy 适合分类）
loss_fn = nn.MSELoss()  # 均方误差

# 优化器
optimizer = optim.Adam(net.parameters(), lr=0.001)
# 也可以: optim.SGD(net.parameters(), lr=0.01, momentum=0.9)

print(f"优化器: {optimizer}")

In [ ]:
# 模拟一轮训练
x = torch.randn(16, 4)          # batch=16, 特征=4
y_true = torch.randn(16, 1)     # 目标值

# 1. 前向传播
y_pred = net(x)

# 2. 计算 loss
loss = loss_fn(y_pred, y_true)

# 3. 清零梯度（否则会累加）
optimizer.zero_grad()

# 4. 反向传播
loss.backward()

# 5. 更新参数
optimizer.step()

print(f"Loss: {loss.item():.4f}")

## 7. 完整训练循环

以下是训练一个回归网络的完整代码。

In [ ]:
# 生成假数据: y = 2*x1 - 3*x2 + 1.5*x3 + 0.5*x4 + noise
torch.manual_seed(42)
X = torch.randn(500, 4)
true_w = torch.tensor([2.0, -3.0, 1.5, 0.5])
true_b = 0.8
Y = X @ true_w + true_b + torch.randn(500) * 0.3
Y = Y.unsqueeze(1)  # (500,) -> (500, 1)

print(f"X 形状: {X.shape}, Y 形状: {Y.shape}")

# 数据集划分
split = 400
X_train, Y_train = X[:split], Y[:split]
X_test, Y_test = X[split:], Y[split:]

In [ ]:
# 构建网络和训练组件
model = SimpleNet(input_dim=4, hidden_dim=32, output_dim=1)
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 200
train_losses = []

for epoch in range(epochs):
    # ---- 训练 ----
    model.train()  # 切换到训练模式（影响 Dropout、BatchNorm）
    
    y_pred = model(X_train)
    loss = loss_fn(y_pred, Y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1:3d},  Train Loss: {loss.item():.4f}")

# ---- 测试 ----
model.eval()  # 切换到评估模式
with torch.no_grad():
    y_test_pred = model(X_test)
    test_loss = loss_fn(y_test_pred, Y_test)
print(f"\nTest Loss: {test_loss.item():.4f}")

In [ ]:
# 绘制训练曲线
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(train_losses, linewidth=1)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.grid(True, alpha=0.3)
plt.show()

## 8. 保存与加载模型

In [ ]:
# 保存（推荐保存 state_dict，而非整个模型）
torch.save(model.state_dict(), "/tmp/model_weights.pt")

# 加载
loaded_model = SimpleNet(input_dim=4, hidden_dim=32, output_dim=1)
loaded_model.load_state_dict(torch.load("/tmp/model_weights.pt"))
loaded_model.eval()

# 验证加载前后输出一致
with torch.no_grad():
    out1 = model(X_test[:3])
    out2 = loaded_model(X_test[:3])
print("加载前后一致:", torch.allclose(out1, out2))

## 9. DataLoader 批处理

`Dataset` + `DataLoader` 负责数据加载、打乱、分批。

In [ ]:
from torch.utils.data import Dataset, DataLoader

# 自定义 Dataset
class MyDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# 创建 DataLoader
train_dataset = MyDataset(X_train, Y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 用 DataLoader 训练的简化写法
print(f"共 {len(train_loader)} 个 batch\n")

for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
    if batch_idx >= 2:  # 只演示前两个 batch
        break
    print(f"Batch {batch_idx}: x 形状={batch_x.shape}, y 形状={batch_y.shape}")

## 10. 实战：用网络拟合 Q 函数

RL 中最常见的场景：用神经网络代替 Q 表，输入状态，输出所有动作的 Q 值。
这里用随机数据演示网络结构。

In [ ]:
class QNetwork(nn.Module):
    """Q 网络：输入 state (4维)，输出每个 action 的 Q 值 (2维)"""
    def __init__(self, state_dim=4, action_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
        )
    
    def forward(self, state):
        return self.net(state)

q_net = QNetwork()

# 模拟：batch=8 个状态
states = torch.randn(8, 4)
q_values = q_net(states)

print(f"输入状态形状: {states.shape}")
print(f"输出 Q 值形状: {q_values.shape}")  # (8, 2)
print("\nQ 值 (每行是一个状态的各动作 Q 值):")
print(q_values)

# 取每个状态的最优动作
best_actions = q_values.argmax(dim=1)
print(f"\n每个状态的最优动作: {best_actions}")

## 11. 实战：Policy Network

Policy Gradient 中，网络输出动作的概率分布。

In [ ]:
class PolicyNetwork(nn.Module):
    """策略网络：输入 state，输出每个 action 的概率"""
    def __init__(self, state_dim=4, action_dim=2):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
        )
    
    def forward(self, state):
        logits = self.fc(state)
        probs = torch.softmax(logits, dim=-1)  # 转换为概率
        return probs

policy = PolicyNetwork()

states = torch.randn(5, 4)
action_probs = policy(states)

print("动作概率 (每行和为 1):")
for i in range(5):
    print(f"  State {i}: {action_probs[i].detach().numpy()}")

# 按概率采样动作
sampled = torch.multinomial(action_probs, num_samples=1)
print(f"\n采样动作: {sampled.squeeze()}")

## 总结

| 主题 | 关键 API |
|------|----------|
| Tensor | `torch.tensor`, `rand/randn/zeros/ones/arange`, `.reshape/.T/.to` |
| 运算 | `+/-/*/@`, 广播, 索引切片, 聚合 `.sum/.mean/.max` |
| 设备 | `.to(device)`, `.cpu()`, `torch.cuda.is_available()` |
| Autograd | `requires_grad=True`, `.backward()`, `.grad`, `torch.no_grad()` |
| nn.Module | `nn.Linear`, `nn.ReLU`, `nn.Sequential`, `forward()` |
| 训练 | `loss_fn`, `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` |
| 保存加载 | `torch.save(state_dict)`, `model.load_state_dict()` |
| DataLoader | `Dataset`, `DataLoader(batch_size, shuffle)` |
| RL 应用 | Q 网络(输出 Q 值), Policy 网络(输出概率+softmax+multinomial) |

掌握以上内容后，即可阅读 DQN 和 REINFORCE 的 PyTorch 实现。